# IMPORTS & UTILITIES

## IMPORTS

In [ ]:
import numpy as np

import plotly.graph_objs as go
import matplotlib.pyplot as plt
from scipy.stats import chi2
import os
import cv2
from IPython.display import Video



## DATA MANIPOLATION

In [ ]:
def get_data(file_name):
    return np.array([np.array([row]).T for row in np.genfromtxt(file_name, delimiter=',')])

def confidence_radius(matrices, confidence_level=0.9999):
    values = []
    for matrix in matrices:
        chi2_critical_value = chi2.ppf(confidence_level, df=3)

        # Compute the eigenvalues of the covariance matrix
        eigenvalues = np.linalg.eigvals(matrix)

        # Compute the maximum radius of the ellipsoid
        max_radius = np.sqrt(np.real(chi2_critical_value * np.max(eigenvalues))) 
        values.append(max_radius)
    return np.array(values)

def evaluate_model(predicted,correct,radius_arr):
   dists = np.linalg.norm(predicted - correct, axis=1)
   
   print(f" Mean radius of the 99.9% perc:      {np.mean(radius_arr):.3f}")
   print(f" Predictions outside the 99.9% perc: {np.sum(dists>radius_arr)/len(predicted)*100:.3f}")
   print(f" Mean error:                         {np.mean(dists):.3f}")

## FILE MANIPOLATION

In [ ]:
def save_video(name, array_frame, fps=2):
    height, width, _ = array_frame[0].shape
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    video = cv2.VideoWriter(name, fourcc, fps, (width, height))

    for frame in array_frame:
        video.write(frame)

    video.release()
    return name

def plot_sphere(ax, center, radius, color='r', alpha=0.3, resolution=50):
    u = np.linspace(0, 2 * np.pi, resolution)  # Azimuthal angle
    v = np.linspace(0, np.pi, resolution)     # Polar angle
    x = radius * np.outer(np.sin(v), np.cos(u)) + center[0]
    y = radius * np.outer(np.sin(v), np.sin(u)) + center[1]
    z = radius * np.outer(np.cos(v), np.ones_like(u)) + center[2]
    
    ax.plot_surface(x, y, z, color=color, alpha=alpha, edgecolor='none')

def frames(positions, radii, step=1, trace_length= 20, remove=True,correct =None):
    output_dir = "imgs"
    os.makedirs(output_dir, exist_ok=True)

    positions = np.array(positions)[::step]  # Downsample positions
    correct = np.array(correct)[::step]  # Downsample positions
    radii = np.array(radii)[::step]          # Downsample radii
    N = len(positions)

    # Compute the axis limits
    max_radius = np.max(radii)  # Find the largest sphere radius
    lim_x = np.max(np.abs(positions[:, 0]))+max_radius
    lim_y = np.max(np.abs(positions[:, 1]))+max_radius
    lim_z = np.max(np.abs(positions[:, 2]))+max_radius

    images = []

    for i in range(N):
        print(f"{str(round(i / N * 100, 2))} % completed")

        # Create figure and 3D axis
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')

        # Plot current spaceship position and its sphere
        current_position = positions[i]
        current_radius = radii[i]

        ax.scatter(current_position[0], current_position[1], current_position[2],
                   color='r', label='Spaceship\'s predicted position', s=5)
        plot_sphere(ax, center=current_position, radius=current_radius, color='r', alpha=0.3)

        # Plot trace (previous positions)
        start_trace = max(0, i - trace_length)
        trace_positions = positions[start_trace:i]
        ax.scatter(trace_positions[:, 0], trace_positions[:, 1], trace_positions[:, 2],
                   color='r', alpha=0.2, s=5)

        if correct is not None:
            ax.scatter(correct[i,0], correct[i,1], correct[i,2],
                   color='black', label='Spaceship\'s true position', s=5)
            
            # Plot trace (previous positions)
            start_trace = max(0, i - trace_length)
            trace_positions = correct[start_trace:i]
            ax.scatter(trace_positions[:, 0], trace_positions[:, 1], trace_positions[:, 2],
                    color='black', alpha=0.2, s=5)


        # Set axis limits
        ax.set_xlim(-lim_x, lim_x)
        ax.set_ylim(-lim_y, lim_y)
        ax.set_zlim(-lim_z, lim_z)

        # Remove labels and ticks
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])

        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_zlabel('')

        # Title
        ax.set_title(f"Position Predictions & Accuracy")
        ax.legend()

        # Save the current frame
        filename = f"{output_dir}/{i}.png"
        plt.savefig(filename)
        plt.close(fig)

        # Read and store the image
        image = cv2.imread(filename)
        images.append(image)

        # Remove the saved file if needed
        if remove:
            os.remove(filename)

    return images

## KALMAN FILTERING

In [ ]:

class Kfilter():

    def check_dimensions(matrix,I,J):
        matrix = np.array(matrix,dtype='float64')
        if matrix.shape != (I,J):
            raise Exception(f"Invalid size for matrix \n{matrix}\nExpected {I},{J}")
        return matrix
        
    def __init__(self,H,R,Q,PHI):

        n = len(PHI)
        m = len(H)

        self.R =   Kfilter.check_dimensions(R,m,m)
        self.Q =   Kfilter.check_dimensions(Q,n,n)
        self.H =   Kfilter.check_dimensions(H,m,n)
        self.PHI = Kfilter.check_dimensions(PHI,n,n)

        self.history = None

    def apply_filter(
            self,
            real_dataset,
            noisy_dataset,
            S_init = None,
            P_init = None
        ):
        #expected dimension
        n = self.PHI.shape[0]
        real_dataset =  [Kfilter.check_dimensions(data,n,1) for data in real_dataset ]
        noisy_dataset = [Kfilter.check_dimensions(data,n,1) for data in noisy_dataset]

        S_ = noisy_dataset[0].copy()              if S_init is None else S_init.copy()
        P_ = np.ones((n,n),dtype='float64')*1000000 if P_init is None else P_init.copy()      

        H = self.H
        R = self.R
        PHI = self.PHI
        I = np.eye(n)
        Q = self.Q
    
        history =[]
        #study all the measurements
        for real_state,noisy_state in zip(real_dataset,noisy_dataset):
            K =  P_ @ H.T @ np.linalg.inv(H @ P_ @ H.T + R)

            m =   H @ noisy_state
            inn = m - H @ S_

            P_ = (I - K @ H) @ P_
            S_ = S_ + K @ inn

            P_estim = (PHI @ P_ @ PHI.T) + Q
            S_estim = PHI @ S_
        
            history.append(Kstate(
                real_state,m,
                inn,K.copy(),
    	        S_.copy(),
                P_.copy()
            ))

            P_ = P_estim
            S_ = S_estim
        
        self.history = history
        return

    def get_values(self,field,code = None):
        return np.array([data.get_field(field,code) for data in self.history],dtype="float64")
    
    def process_data(self, data_type, split=False):
        # Map data type to component symbols
        component_map = {
            "Position": "p",
            "Velocity": "v",
            "Acceleration": "a"
        }
        
        # Fetch component symbol
        component = component_map.get(data_type)
        if not component:
            raise ValueError(f"Invalid data type '{data_type}'. Choose from 'Position', 'Velocity', 'Acceleration'.")
        
        # Retrieve posteriori state and variance
        belief = self.get_values("posteriori_state", component)
        radius_arr = confidence_radius(self.get_values("posteriori_variance", component))
        
        # Retrieve correct real states from the reference model
        correct_states = self.get_values("real_state", component)
        # Plot data
        
        plot_data(
            belief,
            f"{data_type}s",
            radius_arr,
            reference_samples=correct_states,
            split=split
        )
        
        # Plot error
        plot_error(
            belief,
            correct_states,
            radius_arr,
            title=f"{data_type}s"
        )
        
        # Evaluate model and print result
        return evaluate_model(belief, correct_states, radius_arr)


In [ ]:
class Kstate():
    def __init__(self,
            real_state,
            measurement,
            innovation,
            kalman_gain,
            posteriori_state,
            posteriori_variance
        ):
        #what is really happening
        self.real_state = np.array(real_state)
        self.measurement = np.array(measurement)
        #updating our belief
        self.innovation = np.array(innovation)
        self.kalman_gain = np.array(kalman_gain)
        #updated belief
        self.posteriori_state = np.array(posteriori_state)
        self.posteriori_variance = np.array(posteriori_variance)

    def get_field(self, field_name,code = None):
        # Check if the attribute exists
        if hasattr(self, field_name):
            data = getattr(self, field_name)
        else:
            raise AttributeError(f"Field '{field_name}' does not exist in Kstate.")
        
        if code is None:
            return data
        
        idx = {
            "p": [0, 1, 2],
            "v": [3, 4, 5], 
            "a": [6, 7, 8]
        }.get(code[0])
        if not idx: raise ValueError("Invalid code prefix. Must be 'p', 'v', or 'a'.")

        if len(code) > 1:
            axis = {
                "x": 0, 
                "y": 1, 
                "z": 2
            }.get(code[1])
            if axis is None:
                raise ValueError("Invalid code suffix. Must be 'x', 'y', or 'z'.")
            idx = [idx[axis]]
        
        if (data.shape[1] == 1):
            return np.array(data[idx, 0], dtype="float64")
        else:
            return np.array(data[np.ix_(idx, idx)], dtype="float64")
    


## PLOTS

In [ ]:
def plot_data(noisy_samples, title, radius_arr=None, reference_samples=None, split=False):
    global d_t
    # Compute the time domain based on the number of samples and time step
    domain = [i * d_t for i in range(len(noisy_samples))]
    
    # Axis labels and colors for x, y, z
    axis_labels = ["x", "y", "z"]
    colors = ["r", "g", "b"]

    if split:
        # Create a 3-row subplot if split=True
        fig, axes = plt.subplots(3, 1, figsize=(15, 6), sharex=True)
        
        # Loop through each axis (x, y, z) and plot them in separate subplots
        for i, (label, color) in enumerate(zip(axis_labels, colors)):
            position = noisy_samples[:, i]
            
            # Plot noisy data
            axes[i].plot(domain, position, label=f"{title} {label}", color=color)
            
            # Plot variances as uncertainty bands if provided
            if radius_arr is not None:
                axes[i].fill_between(
                    domain, position - radius_arr, position + radius_arr, 
                    color=color, alpha=0.2, label=f"Uncertainty ({label.upper()})"
                )
            
            # Plot reference data if provided
            if reference_samples is not None:
                reference = reference_samples[:, i]
                axes[i].plot(
                    domain, reference, color=color, linestyle='dashed', alpha=0.5
                )
            
            # Set titles, labels, and legends
            axes[i].set_title(f"{title} {label.upper()}")
            axes[i].set_xlabel("Time (s)")
            axes[i].set_ylabel(f"{title}")
            axes[i].legend()
            axes[i].grid()
    else:
        # Prepare the figure for a single plot if split=False
        plt.figure(figsize=(15, 6))
        
        # Loop through each axis (x, y, z) and plot them in a single plot
        for i, (label, color) in enumerate(zip(axis_labels, colors)):
            position = noisy_samples[:, i]
            
            # Plot noisy data
            plt.plot(domain, position, label=f"{title} {label}", color=color)
            
            # Plot variances as uncertainty bands if provided
            if radius_arr is not None:
                plt.fill_between(
                    domain, position - radius_arr, position + radius_arr, 
                    color=color, alpha=0.2, label=f"Uncertainty ({label.upper()})"
                )
            
            # Plot reference data if provided
            if reference_samples is not None:
                reference = reference_samples[:, i]
                plt.plot(
                    domain, reference, color=color, linestyle='dashed', alpha=0.5
                )

        # Add title, labels, and legend for single plot
        plt.title(f"{title} - All Components")
        plt.xlabel("Time (s)")
        plt.ylabel(f"{title}")
        plt.legend()
        plt.grid()

    # Adjust layout for multiple subplots
    plt.tight_layout()
    # Show the plot
    plt.show()


In [ ]:
def plot_error(predicted, correct, radius_arr, title="Position"):
    global d_t
    # Compute the time domain based on the number of samples and time step
    domain = [i * d_t for i in range(len(predicted))]

    # Calculate the distance (error) between the predicted and correct values
    dists = np.linalg.norm(predicted - correct, axis=1)

    # Prepare the figure
    plt.figure(figsize=(15, 6))

    # Plot the error (distance between predicted and correct values)
    plt.plot(domain, dists, label="Error (Predicted vs Correct)", color='black')

    # Plot the radius (variance) to visualize the system's uncertainty
    plt.plot(domain, radius_arr, label="System Variance (Radius)", color='orange', linestyle='dashed')

    # Add title, labels, and legend
    plt.title(f"Prediction Error on {title} ")
    plt.xlabel("Time (s)")
    plt.ylabel("Distance")
    plt.legend()
    plt.grid()

    # Show the plot
    plt.tight_layout()
    plt.show()


# PRE-PROCESSING DATA

Let's now load the data we just create with the simulation.

Since we have a lot of data, we sample only 1/5, to not "overload" the following data visualization.

In [ ]:
original_d_t = 0.01
step = 5
d_t = original_d_t * step

correct_data = get_data('../data/orbiting.csv')[::step]
noisy_data = get_data('../data/orbiting_noisy.csv')[::step]

correct_positions = correct_data[:,[0,1,2],0]
correct_velocities = correct_data[:,[3,4,5],0]
correct_accelerations = correct_data[:,[6,7,8],0]

print(f"Total Points: {len(correct_data)*step}")
print(f"d_t: {d_t} s")
print(f"Measured point: {len(correct_data)}")

# KALMAN FILTERS

We now compute kalman filtering, hence updating the predicted state and variance:






1. **Update**

    $ K_t = P_{t}^{-} H^\top \left(H P_{t}^{-} H^\top + R\right)^{-1} $

    $ P_{t} = \left(I - K_t H\right) P_{t}^{-} $

    $ \hat{S}_{t} = \hat{S}_{t}^{-} + K_t \left(m_t - H_t \hat{S}_{t}^{-}\right) $



2. **Projection Estimate**

    $ \hat{S}_{t+1}^{-} = \Phi \hat{S}_{t} $
    
    $ P_{t+1}^{-} = \Phi P_{t} \Phi^\top + Q_t $


First, let's start fix the dimensionality and some matrices of the problem.
- $n = 9$ (3 axis for position, velocity and acceleration)
- $m = 3$ (only the position is measured)
- $
H = \begin{bmatrix}
1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 1 & 0 & 0 & 0 & 0 & 0 & 0
\end{bmatrix}
$

- $
R = \begin{bmatrix}
\sigma^2 & 0 & 0 \\
0 & \sigma^2 & 0 \\
0 & 0 & \sigma^2
\end{bmatrix}
$  $ \sigma = 1000 $ noise of the measurements


In [ ]:
H = [
    [1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 0]
]

#correct std from the noise 
std_var = 1000 

R = np.diag(
    [std_var**2,std_var**2,std_var**2]
)

## MODEL 1

Let's  **FIX** the "transition matrix", to describe the **linear process model**.

$$
\Phi = \begin{bmatrix}
1 & 0 & 0 &  0.05 & 0 & 0 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 &  0.05 & 0 & 0 & 0 & 0 \\
0 & 0 & 1 & 0 & 0 &  0.05 & 0 & 0 & 0 \\
0 & 0 & 0 & 1 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 1 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 1 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 1 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 1
\end{bmatrix}
$$


- The position is linearly updated by the velocity
- The velocity is constant (but updates the position, ence is "considered" by the filter).
- The acceleration (i.e. Gravitational Pull) is constant, and independent from all the others coefficients of the State.


In [ ]:
PHI = [
    [ 1 , 0 , 0 ,d_t, 0 , 0 , 0 , 0 , 0],
    [ 0 , 1 , 0 , 0 ,d_t, 0 , 0 , 0 , 0],
    [ 0 , 0 , 1 , 0 , 0 ,d_t, 0 , 0 , 0],
        
    [ 0 , 0 , 0 , 1 , 0 , 0 , 0 , 0 , 0],
    [ 0 , 0 , 0 , 0 , 1 , 0 , 0 , 0 , 0],
    [ 0 , 0 , 0 , 0 , 0 , 1 , 0 , 0 , 0],
        
    [ 0 , 0 , 0 , 0 , 0 , 0 , 1 , 0 , 0],
    [ 0 , 0 , 0 , 0 , 0 , 0 , 0 , 1 , 0],
    [ 0 , 0 , 0 , 0 , 0 , 0 , 0 , 0 , 1]
]

We now only have to define $Q$, the covariance matrix who descrive the "interinsic randomness" of the model.

In other words, it define "how much" of the **Real** Transition Matrix we have left, masking it as "noise". 

### UNDERFITTING MODEL

Let's now try a model with "low" values of $Q$.

$$
Q = \begin{bmatrix}
500 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 500 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 500 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 500 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 500 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 500 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 500 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 500 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 500
\end{bmatrix}
$$

In other words, we are saying to the filter: 

**"Don't trust the data, the transition matrix is perfect as it is"**

In [ ]:
Q = np.diag([500] * 9)

model_1a = Kfilter(H,R,Q,PHI)

model_1a.apply_filter(correct_data,noisy_data)

In [ ]:
%%script false --no-raise-error
save_video("../videos/model_1A.mp4",frames(
    model_1a.get_values("posteriori_state", "p"),
    confidence_radius(model_1a.get_values("posteriori_variance", "p")),
    correct=model_1a.get_values("real_state", "p"),
    step = 3)
    ,30)

In [ ]:
Video("../videos/model_1A.mp4", embed=True,width=800,height=600)

We can clearly see how the model's predictions are very far from the real values, since it requires a lot of measurements to "adapt" to the new direction.

By showing the separate functions, it's even more clear.

- Observe how the **Uncertainty region** (described by the covariance matrix), is really small: it reflects the fact that the model is **VERY** confident about his predictions.

In [ ]:
model_1a.process_data("Position",split=True)

The stats cofirm indeed that the predictions are (basically) **always** far from the real state.

### OVERFITTING MODEL

Now let's try a model with "high" values of $Q$.

$$
Q = \begin{bmatrix}
10^8 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 10^8 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 10^8 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 10^8 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 10^8 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 10^8 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 10^8 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 10^8 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 10^8
\end{bmatrix}
$$

In other words, we are saying to the filter: 

**"Completely trust the data, the transition matrix is basically useless"**

In [ ]:
Q = np.diag([100000000] * 9)

model_1b = Kfilter(H,R,Q,PHI)
model_1b.apply_filter(correct_data,noisy_data)

In [ ]:
%%script false --no-raise-error

save_video("../videos/model_1B.mp4",frames(
    model_1b.get_values("posteriori_state", "p"),
    confidence_radius(model_1b.get_values("posteriori_variance", "p")),
    correct=model_1b.get_values("real_state", "p"),
    step = 3)
    ,30)

In [ ]:
Video("../videos/model_1B.mp4", embed=True,width=800,height=600)

In [ ]:
model_1b.process_data("Position")

We can observe of the mean error now is much more low (before was 6172.238), and the **State's covariance** fully "inglobe" all the predictions.

It seem's too good to be true, where is the catch?

In [ ]:
model_1b.process_data("Velocity")

The velocity plots are associated with a **very large** region, hence the model has no idea of the correct values of those state's coefficient.

So, even this is not the correct solution to our problem: what is in the middle? 

### BEST MODEL

By doing a brute-force to find the best values for the matrix $Q$, we have found:
$$
Q = \begin{bmatrix}
10 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 10 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 10 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 545560 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 545560 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 545560 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0
\end{bmatrix}
$$

In [ ]:
Q = np.diag(np.repeat(
    [10,545560,0]
,3))

model_1c = Kfilter(H,R,Q,PHI)

model_1c.apply_filter(correct_data,noisy_data)

In [ ]:
%%script false --no-raise-error

save_video("../videos/model_1C.mp4",frames(
    model_1c.get_values("posteriori_state", "p"),
    confidence_radius(model_1c.get_values("posteriori_variance", "p")),
    correct=model_1c.get_values("real_state", "p"),
    step = 3)
    ,30)

In [ ]:
Video("../videos/model_1C.mp4", embed=True,width=800,height=600)

In [ ]:
model_1c.process_data("Position")

In [ ]:
model_1c.process_data("Velocity",split=True)

As said before we have found a compromise to predict the position and the velocity, but **not the Acceleration**

In [ ]:
model_1c.process_data("Acceleration")

## MODEL 2 

Let's  now change the "transition matrix", to allow the prediction to change his belief about the Acceleration (without compromizing the accuracy on Position and Velocity):

$$
\Phi = \begin{bmatrix}
1 & 0 & 0 &  0.05 & 0 & 0 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 &  0.05 & 0 & 0 & 0 & 0 \\
0 & 0 & 1 & 0 & 0 &  0.05 & 0 & 0 & 0 \\
0 & 0 & 0 & 1 & 0 & 0 & 0.05 & 0 & 0 \\
0 & 0 & 0 & 0 & 1 & 0 & 0 & 0.05 & 0 \\
0 & 0 & 0 & 0 & 0 & 1 & 0 & 0 & 0.05 \\
0 & 0 & 0 & 0 & 0 & 0 & 1 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 1
\end{bmatrix}
$$


- The position is linearly updated by the velocity
- The velocity is linearly updated by the velocity
- The acceleration (i.e. Gravitational Pull) is constant, but can be updated by the filter (since it is considered in the prediction of the next state).

Let's use the $Q$ that better fit the simulation:

$$
Q = \begin{bmatrix}
10 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 10 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 10 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 10 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 10 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 10 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 555560 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 555560 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 555560
\end{bmatrix}
$$


In [ ]:
PHI = [
    [ 1 , 0 , 0 ,d_t, 0 , 0 , 0 , 0 , 0],
    [ 0 , 1 , 0 , 0 ,d_t, 0 , 0 , 0 , 0],
    [ 0 , 0 , 1 , 0 , 0 ,d_t, 0 , 0 , 0],
        
    [ 0 , 0 , 0 , 1 , 0 , 0 ,d_t , 0 , 0],
    [ 0 , 0 , 0 , 0 , 1 , 0 , 0 ,d_t , 0],
    [ 0 , 0 , 0 , 0 , 0 , 1 , 0 , 0 ,d_t],
        
    [ 0 , 0 , 0 , 0 , 0 , 0 , 1 , 0 , 0],
    [ 0 , 0 , 0 , 0 , 0 , 0 , 0 , 1 , 0],
    [ 0 , 0 , 0 , 0 , 0 , 0 , 0 , 0 , 1]
]

Q = np.diag(np.repeat(
    [10,10,555560]
,3))

model_2 = Kfilter(H,R,Q,PHI)

model_2.apply_filter(correct_data,noisy_data)

In [ ]:
%%script false --no-raise-error

save_video("../videos/model_2.mp4",frames(
    model_2.get_values("posteriori_state", "p"),
    confidence_radius(model_2.get_values("posteriori_variance", "p")),
    correct=model_2.get_values("real_state", "p"),
    step = 3)
    ,30)

In [ ]:
Video("../videos/model_2.mp4", embed=True,width=800,height=600)

In [ ]:
model_2.process_data("Position")

As we can see, the model have a pretty good "understanding" of the position of the spaceship (the model has a mean error a little bigger).

But what about the velocity and Gravitational Pull?

In [ ]:
model_2.process_data("Velocity",split=True)
model_2.process_data("Acceleration",split=True)

The main observation we can make is that now the acceleration's predictions about the **hidden state** is no longer fixed, but dinamically changes.

## MODEL N 

In [ ]:
def expand_phi(PHI, K):
    global d_t
   # Size of the original PHI matrix
    original_size = PHI.shape[0]
    
    # New size: original size + 3 * K for rows and columns
    new_size = original_size + 3 * K
    
    # Initialize the expanded matrix with zeros
    expanded_phi = np.zeros((new_size, new_size), dtype='float64')
    
    # Place the original PHI in the top-left corner
    expanded_phi[:original_size, :original_size] = PHI
    
    # Add 3x3 identity blocks scaled by d_t
    for i in range(K):
        block_start_row = original_size + 3 * i
        block_start_col = original_size + 3 * i
        
        # Add diagonal 3x3 block
        expanded_phi[block_start_row:block_start_row + 3, block_start_col:block_start_col + 3] = np.eye(3) * 1
        
        # Add offset block above the diagonal (scaled by d_t)
        if block_start_row - 3 >= 0:
            expanded_phi[block_start_row - 3:block_start_row, block_start_col:block_start_col + 3] = np.eye(3) * d_t
    
    return expanded_phi

def pretty_print(matrix):
    # ANSI escape code for red text
    red = "\033[91m"
    reset = "\033[0m"
    
    for row in matrix:
        # For each value in the row, if it's non-zero, print it in red
        print(" ".join(f"{red}{val: .3f}{reset}" if val != 0 else f"{val: .3f}" for val in row))
    print("\n\n")

In [ ]:
K=1 #lasciare cosi la Q
Q = np.diag(
    np.concatenate([
        np.array([10, 10, 10, 10, 10, 10, 10, 10, 10]),  # Fixed first 9 values
        np.concatenate([np.repeat(10000*10**i, 3) for i in range(K)])  # Dynamically generated values
    ])
) *100


PHI = [
    [ 1 , 0 , 0 ,d_t, 0 , 0 , 0 , 0 , 0],
    [ 0 , 1 , 0 , 0 ,d_t, 0 , 0 , 0 , 0],
    [ 0 , 0 , 1 , 0 , 0 ,d_t, 0 , 0 , 0],
        
    [ 0 , 0 , 0 , 1 , 0 , 0 ,d_t , 0 , 0],
    [ 0 , 0 , 0 , 0 , 1 , 0 , 0 ,d_t , 0],
    [ 0 , 0 , 0 , 0 , 0 , 1 , 0 , 0 ,d_t],
        
    [ 0 , 0 , 0 , 0 , 0 , 0 , 1 , 0 , 0],
    [ 0 , 0 , 0 , 0 , 0 , 0 , 0 , 1 , 0],
    [ 0 , 0 , 0 , 0 , 0 , 0 , 0 , 0 , 1]
]

PHI_exp = expand_phi(np.array(PHI),K)

H = np.array([
    [1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 0],
],dtype='float64')
H_exp = np.array([
    np.concatenate([row,np.zeros(3*K, dtype='float64')]) for row in H
],dtype='float64') 


correct_data_ext = np.array([np.vstack((data,np.zeros((3*K,1)))) for data in correct_data],dtype='float64') 
noisy_data_ext  = np.array([np.vstack((data,np.zeros((3*K,1)))) for data in noisy_data],dtype='float64') 


In [ ]:
model_N = Kfilter(H_exp,R,Q,PHI_exp)

model_N.apply_filter(correct_data_ext,noisy_data_ext,P_init=Q)

In [ ]:
print(model_N.process_data("Position"))
print(model_N.process_data("Velocity",split=True))
print(model_N.process_data("Acceleration",split=True))

## BRUTE-FORCE HYPERP-TUNING (HOW TO FIND BEST Q)

In [ ]:
%%script false --no-raise-error
###CODE DISABLED

PHI = [
    [ 1 , 0 , 0 ,d_t, 0 , 0 , 0 , 0 , 0],
    [ 0 , 1 , 0 , 0 ,d_t, 0 , 0 , 0 , 0],
    [ 0 , 0 , 1 , 0 , 0 ,d_t, 0 , 0 , 0],
        
    [ 0 , 0 , 0 , 1 , 0 , 0 ,d_t, 0 , 0],
    [ 0 , 0 , 0 , 0 , 1 , 0 , 0 ,d_t, 0],
    [ 0 , 0 , 0 , 0 , 0 , 1 , 0 , 0 ,d_t],
        
    [ 0 , 0 , 0 , 0 , 0 , 0 , 1 , 0 , 0],
    [ 0 , 0 , 0 , 0 , 0 , 0 , 0 , 1 , 0],
    [ 0 , 0 , 0 , 0 , 0 , 0 , 0 , 0 , 1]
]

vals = np.logspace(1, 10, 20)
results = []
i = 0
# Iterate over all combinations of a, b, c sampled from vals
for a in vals:
    for b in vals:
        for c in vals:
            # Create Q matrix
            Q = np.diag(np.repeat([a,b,c],3))

            # Create and apply model
            model = Kfilter(H, R, Q, PHI)
            model.apply_filter(correct_data, noisy_data)
            # Save results
            results.append({
                "a": a,
                "b": b,
                "c": c,
                "pos": evaluate_model(model.get_values("posteriori_state", "p"), model.get_values("real_state", "p"), confidence_radius(model.get_values("posteriori_variance", "p"))),
                "vel": evaluate_model(model.get_values("posteriori_state", "v"), model.get_values("real_state", "v"), confidence_radius(model.get_values("posteriori_variance", "v"))),
                "acc": evaluate_model(model.get_values("posteriori_state", "a"), model.get_values("real_state", "a"), confidence_radius(model.get_values("posteriori_variance", "a")))
            })
            i += 1

            print(f"{a:12.6f}, {b:12.6f}, {c:12.6f} : {i / (len(vals)**3) * 100:.6f}%")



In [ ]:
%%script false --no-raise-error
import csv
import ast  # To safely evaluate string representations of Python literals

# Open and read the CSV file
with open('results.csv', 'r') as file:
    reader = csv.DictReader(file)  # Use DictReader for column-based access
    results = []
    
    for row in reader:
        # Parse the string representation of tuples into actual Python tuples
        row['pos'] = ast.literal_eval(row['pos'])
        row['vel'] = ast.literal_eval(row['vel'])
        row['acc'] = ast.literal_eval(row['acc'])
        
        # Convert the rest of the numeric fields
        row['a'] = float(row['a'])
        row['b'] = float(row['b'])
        row['c'] = float(row['c'])
        
        results.append(row)

# Supponiamo che 'threshold' sia la soglia per l'errore massimo accettabile
threshold = 10  # Cambia questo valore secondo necessità

# Filtra i risultati con pos[0], vel[0], acc[0] entro la soglia
filtered_results = [
    result for result in results
    if abs(result["pos"][0]) <= threshold and
       abs(result["vel"][0]) <= threshold and
       abs(result["acc"][0]) <= threshold
]

# Funzione per calcolare la somma dei valori z
def calculate_sum_z(entry):
    pos_z = entry["pos"][2]
    vel_z = entry["vel"][2]
    acc_z = entry["acc"][2]
    return pos_z + vel_z + acc_z

# Trovare il dizionario con la somma minima dai risultati filtrati
if filtered_results:  # Controlla che ci siano risultati dopo il filtro
    min_entry = min(filtered_results, key=calculate_sum_z)
    print("Dizionario con la somma minima dopo il filtro:")
    print(min_entry)
else:
    print("Nessun risultato soddisfa la condizione della soglia.")

# FINAL REMARKS

Let's conclude our empirical study by writing down the principal observations:
1. **Low** coefficients for the covariance matrix **Q** define a big **bias** in the model: a low **kalman gain** mitigates the effects of the innovation.

2. **High** coefficients for the covariance matrix **Q** define a little **bias** in the model: a big **kalman gain** allow the effects of the innovation on the predicted state.

3. Increasing the dimensionality of the hidden state does not improve predictions on the measured coefficients; on the contrary, it induces a larger error on the directly observed values (e.g., position). The amount of information obtained remains the same, but since it is spread across a larger number of coefficients, we lose 'certainty' about the others.

4. **Linear Kalman Filters** find the **best linear solution**, that is the same as the global best solution only if the real noise on the model **Q** is gaussian: since our fluctuation on the acceleration are not equally independent from a graussian distribution, there exist some **better models** (but surely, not linear), such as "Extended Kalman Filter" or "Unscented kalman filter".